In [7]:
import json, sys
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from lightgbm import LGBMRegressor

sys.path.insert(0, "data")
from metric_cv import quantile_quality, cv_json

tr = pd.read_csv("data/train.csv")
te = pd.read_csv("data/test.csv")
sample = pd.read_csv("data/sample_submission.csv")
id_train = tr[["id"]].copy()
id_test = te[["id"]].copy()
y = tr["daily_orders"].to_numpy(float)
tr = tr.drop(columns=["id", "daily_orders"])
te = te.drop(columns="id")
QUANTILES = (0.1, 0.5, 0.9)

PARAM = dict(it=500, lr=0.05)

cat_features = ["city", "district_type", "entrance_type"]
for col in cat_features:
    categories = pd.Index(pd.concat([tr[col], te[col]], ignore_index=True).dropna().unique())
    tr[col] = pd.Categorical(tr[col], categories=categories)
    te[col] = pd.Categorical(te[col], categories=categories)

def predict(X_train, y_train, X_pred):
    pred = np.zeros((len(X_pred), 3))

    for i, q in enumerate(QUANTILES):
        model = LGBMRegressor(
            objective="quantile",
            alpha=q,
            n_estimators=PARAM["it"],
            learning_rate=PARAM["lr"],
            verbosity=-1,
            random_state=0,
            n_jobs=-1
        )
        model.fit(X_train, y_train, categorical_feature=cat_features)
        pred[:, i] = model.predict(X_pred)

    pred = np.clip(pred, 0, 5000)
    pred.sort(axis=1)
    return pred


splits = list(KFold(4, shuffle=True, random_state=0).split(tr))
oof = np.zeros((len(tr), 3))

for a, b in splits:
    oof[b] = predict(tr.iloc[a], y[a], tr.iloc[b])

folds = [
    100 * quantile_quality(
        y[b], oof[b, 0], oof[b, 1], oof[b, 2], tr["city"].iloc[b]
    )
    for _, b in splits
]

test = np.round(predict(tr, y, te), 1)

sub = pd.DataFrame({
    "id": id_test["id"],
    "predicted_daily_orders_p10": test[:, 0],
    "predicted_daily_orders_p50": test[:, 1],
    "predicted_daily_orders_p90": test[:, 2]
})

assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample)
assert set(sub["id"]) == set(sample["id"])
assert (sub.iloc[:, 1:].to_numpy() >= 0).all()

sub.to_csv("submission.csv", index=False)

cv = cv_json("lgbm_quantile", [f / 100 for f in folds])
print(json.dumps(cv))

{"name": "lgbm_quantile", "metric": "quantile_quality_x100", "folds": [73.62, 75.6, 75.11, 75.4], "cv": 74.93, "std": 0.78}
